# ExIt capacity A/B — embed_dim 128 vs 256 (Gumbel regime)

Answers the long-standing "does a bigger net help?" question **in the regime we
actually ship**: both arm configs (`configs/v2_exit_embed128/256.yaml`) now run
with `gumbel_search: true` (Build 1, +9.4% gate win), identical except
`model.embed_dim`. Both arms are fresh **BC -> ExIt** (no warm start — a 256-dim
model can't load the 128-dim champion), sharing ONE demo cache on Drive so the
demonstration set is byte-identical.

**Parallelizing across two sessions:**
1. Session A: set `ARM='128'` and run — it collects + caches the apex demos
   first (the cache write is logged early, during BC).
2. Session B: once `demos_apex_exit_ab.pkl` exists on Drive, set `ARM='256'`
   and run (the cell checks and refuses to start without the cache).

Or run `ARM='both'` sequentially in one session. GPU runtime recommended.

**Compare the arms to each other** (gate cell below), not to the champion — they
are not warm-started, so cross-comparing to `v2_exit_gumbel_on_a100` conflates
capacity with the warm start.


In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
for sub in ['', '/checkpoints', '/logs', '/submissions']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)

REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT if needed
REPO_DIR = '/content/orbit_wars'
if os.path.exists(REPO_DIR):
    print('Repo present, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

os.system('pip install -q --upgrade "kaggle-environments>=1.28.0" pyyaml tensorboard')

CPU_COUNT = os.cpu_count() or 8

# Silence kaggle_environments' noisy OpenSpiel INFO logs in THIS kernel
# (subprocess training output is filtered by grep on the launch line instead).
import logging
for _ln in ('kaggle_environments.envs.open_spiel_env.open_spiel_env',
            'kaggle_environments.core_harness'):
    logging.getLogger(_ln).disabled = True

print(f'\nDrive: {DRIVE_OUTPUT} | cwd: {os.getcwd()} | vCPUs: {CPU_COUNT}')


In [ ]:
#@title 2. Run the A/B (one arm per session to parallelize)
import os

ARM = 'both'  #@param ['both', '128', '256']

demo_cache = f'{DRIVE_OUTPUT}/demos_apex_exit_ab.pkl'
if ARM == '256':
    assert os.path.exists(demo_cache), (
        f'demo cache missing at {demo_cache} — run the 128 arm first '
        '(it collects + caches the shared demos)')
only = '' if ARM == 'both' else f'--only {ARM}'

!python scripts/run_embed_ab.py {only} \
    --save-dir {DRIVE_OUTPUT}/checkpoints \
    --log-dir {DRIVE_OUTPUT}/logs \
    --demo-cache {demo_cache} \
    --collect-workers {CPU_COUNT} --search-workers {CPU_COUNT} 2>&1 | grep -vE --line-buffered "^\[kaggle_environments\.(envs\.open_spiel_env|core_harness)"


In [ ]:
#@title 3. Gate: 128 vs 256 (side-alternated, multi-seed; per-arm configs)
from pathlib import Path
from v2.config import load_v2_config, v2_config_to_dict
from scripts.eval_fast import _eval_checkpoint

# NOTE: each arm MUST be scored with its own config (the model dims differ).
RUNS = {'v2_exit_embed128': 'configs/v2_exit_embed128.yaml',
        'v2_exit_embed256': 'configs/v2_exit_embed256.yaml'}
ITER         = 'last'                  #@param {type:"string"}  # 'last' or e.g. '25'
GAMES        = 60                      #@param {type:"integer"} # per seed batch
SEED_BATCHES = [20000, 31000, 42000]
EVAL_WORKERS = CPU_COUNT

ckpt_root = Path(f'{DRIVE_OUTPUT}/checkpoints')
ckpt_name = 'ckpt_last.pt' if ITER == 'last' else f'ckpt_{int(ITER):06d}.pt'

print(f'iter={ITER}  games/seed={GAMES}  seeds={SEED_BATCHES}  workers={EVAL_WORKERS}\n')
header = f'{"run":>20} | ' + ' '.join(f'seed{s:>6}' for s in SEED_BATCHES) + f' | {"mean":>5}'
print(header); print('-' * len(header))
results = {}
for run, cfg_path in RUNS.items():
    path = ckpt_root / run / ckpt_name
    if not path.exists():
        print(f'{run:>20} | (no checkpoint at {run}/{ckpt_name} — train it first)')
        continue
    cfg_dict = v2_config_to_dict(load_v2_config(cfg_path))
    wrs = []
    for s in SEED_BATCHES:
        win, loss, tie = _eval_checkpoint(path, cfg_dict, GAMES, EVAL_WORKERS, 'apex', s)
        wrs.append(win / max(win + loss + tie, 1))
    results[run] = wrs
    print(f'{run:>20} | ' + ' '.join(f'{r:>9.0%}' for r in wrs) + f' | {sum(wrs)/len(wrs):>5.0%}')

if len(results) == 2:
    a, b = results['v2_exit_embed128'], results['v2_exit_embed256']
    d = [y - x for x, y in zip(a, b)]
    print('\n256 vs 128, per-seed delta: ' + ' '.join(f'{x:+.0%}' for x in d)
          + f'   mean {sum(d)/len(d):+.1%}   '
          + ('capacity WINS on every seed' if all(x > 0 for x in d)
             else ('capacity LOSES on every seed' if all(x < 0 for x in d) else 'mixed')))
